# 03. MLP 784 -> 128 -> 10: цепное правило через скрытый слой

В ноутбуке 02 линейная модель уперлась в 92.4% и дальше не идет. Причина
разобрана там же: один шаблон на класс не покрывает варианты написания цифры.

Здесь добавляем **скрытый слой** и получаем около 97.5% на тех же данных.
Главное новое в математике - обратный проход впервые идет через нелинейность,
то есть цепное правило работает по-настоящему, а не в один шаг.

## Архитектура

$$Z_1 = X W_1 + b_1, \qquad H_1 = \mathrm{ReLU}(Z_1), \qquad Z_2 = H_1 W_2 + b_2$$

| Тензор | Форма | Что это |
|---|---|---|
| $X$ | $(B, 784)$ | батч картинок |
| $W_1, b_1$ | $(784, 128)$, $(128,)$ | первый слой |
| $Z_1, H_1$ | $(B, 128)$ | скрытый слой до и после ReLU |
| $W_2, b_2$ | $(128, 10)$, $(10,)$ | второй слой |
| $Z_2$ | $(B, 10)$ | логиты |

Параметров: $784 \cdot 128 + 128 + 128 \cdot 10 + 10 = 101\,770$,
то есть в 13 раз больше, чем у линейной модели.

## Почему ReLU между слоями обязательна

Без нелинейности два линейных слоя подряд **эквивалентны одному**:

$$(X W_1 + b_1) W_2 + b_2 = X \underbrace{(W_1 W_2)}_{\text{одна матрица } (784,10)} + \underbrace{(b_1 W_2 + b_2)}_{\text{один вектор } (10,)}$$

Произведение двух матриц - снова матрица, поэтому такая "глубокая" сеть
имела бы ровно ту же выразительную силу, что модель из ноутбука 02, только
с лишними параметрами. ReLU - это то, что вообще делает сеть нелинейной.

$$\mathrm{ReLU}(z) = \max(0, z)$$

Функция тривиальная, но ее хватает: кусочно-линейными функциями с достаточным
числом изломов можно приблизить любую непрерывную функцию.

**Почему именно ReLU, а не сигмоида.** У сигмоиды производная не превышает
0.25 и стремится к нулю на хвостах, поэтому в глубоких сетях градиент при
обратном проходе затухает. У ReLU производная равна ровно 1 на всей
положительной полуоси - градиент проходит без ослабления. Плюс она дешевле:
одно сравнение против экспоненты.

In [ ]:
import math
from pathlib import Path
import gzip
import pickle

import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)
GEN = torch.Generator().manual_seed(0)
print("torch", torch.__version__)

---
## 1. Данные и функции из ноутбука 02

Повторяем загрузку и уже разобранные функции, чтобы ноутбук оставался
самодостаточным и запускался в Colab отдельно от остальных.
Подробные объяснения к ним - в ноутбуке 02.

In [ ]:
DATA_PATH = Path("data")
PATH = DATA_PATH / "mnist"
PATH.mkdir(parents=True, exist_ok=True)
FILENAME = "mnist.pkl.gz"

if not (PATH / FILENAME).exists():
    import requests
    URL = "https://github.com/pytorch/tutorials/raw/main/_static/"
    print("скачиваем", FILENAME, "...")
    (PATH / FILENAME).open("wb").write(requests.get(URL + FILENAME, timeout=60).content)

with gzip.open((PATH / FILENAME).as_posix(), "rb") as f:
    (x_train, y_train), (x_valid, y_valid), _ = pickle.load(f, encoding="latin-1")

x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.int64)
x_valid = torch.tensor(x_valid, dtype=torch.float32)
y_valid = torch.tensor(y_valid, dtype=torch.int64)

n, d = x_train.shape
num_classes = 10
print(f"train {tuple(x_train.shape)}, valid {tuple(x_valid.shape)}")

In [ ]:
# --- прямой проход (разобрано в ноутбуке 02) ---

def softmax(z):
    z_shifted = z - z.max(dim=-1, keepdim=True).values
    e = z_shifted.exp()
    return e / e.sum(dim=-1, keepdim=True)


def log_softmax(z):
    m = z.max(dim=-1, keepdim=True).values
    zs = z - m
    return zs - zs.exp().sum(dim=-1, keepdim=True).log()


def nll(log_probs, y):
    return -log_probs[range(y.shape[0]), y].mean()


def cross_entropy(z, y):
    return nll(log_softmax(z), y)


def accuracy(z, y):
    return (torch.argmax(z, dim=1) == y).to(torch.float32).mean()


def one_hot(y, num_classes=10):
    out = torch.zeros(y.shape[0], num_classes, dtype=torch.float32)
    out[range(y.shape[0]), y] = 1.0
    return out


# --- обратный проход (разобрано в ноутбуке 02) ---

def grad_logits(z, y):
    # dL/dz = (softmax(z) - Y) / B
    p = softmax(z.detach())
    return (p - one_hot(y, z.shape[1]).to(z.dtype)) / y.shape[0]


def grad_linear(x, w, grad_z):
    # z = x @ w + b  ->  (dL/dw, dL/db, dL/dx)
    grad_w = x.detach().T @ grad_z
    grad_b = grad_z.sum(dim=0)
    grad_x = grad_z @ w.detach().T
    return grad_w, grad_b, grad_x


def iterate_minibatches(x, y, batch_size, shuffle=True, generator=None):
    total = x.shape[0]
    order = torch.randperm(total, generator=generator) if shuffle else torch.arange(total)
    for start in range(0, total, batch_size):
        idx = order[start:start + batch_size]
        yield x[idx], y[idx]


print("функции из ноутбука 02 определены")

---
## 2. ReLU и ее производная

$$\mathrm{ReLU}(z) = \max(0, z), \qquad \mathrm{ReLU}'(z) = \begin{cases} 1, & z > 0 \\ 0, & z < 0 \end{cases}$$

В точке $z = 0$ производной формально нет (слева 0, справа 1). Договоренность
и в PyTorch, и у нас - считать ее нулем. На обучение это не влияет:
попадание ровно в нуль на `float32` практически невероятно.

**Ключевое отличие от линейного слоя.** Производная ReLU **поэлементная**:
выход $H_{1[n,j]}$ зависит только от входа $Z_{1[n,j]}$ и ни от чего больше.
Поэтому цепное правило здесь - не матричное умножение, а поэлементное
умножение на маску:

$$\frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial H_1} \odot [Z_1 > 0]$$

Читается буквально: **там, где нейрон был закрыт, градиент не проходит**.

Отсюда известный эффект "мертвых нейронов": если нейрон ушел в отрицательную
зону на всех примерах, его маска всюду нулевая, градиент до его весов не
доходит, и он уже никогда не оживет.

In [ ]:
def relu(z):
    # max(0, z) через clamp - видно, что это просто обрезка снизу нулем
    return z.clamp(min=0.0)


def relu_mask(z):
    # Производная ReLU по ее входу: 1 где z > 0, иначе 0.
    # Приводим к типу z, чтобы маску можно было умножать без приведения.
    return (z > 0).to(z.dtype)


zz = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
print("z         :", zz.tolist())
print("relu(z)   :", relu(zz).tolist())
print("relu'(z)  :", relu_mask(zz).tolist(), "<- в нуле считаем 0")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
grid = torch.linspace(-3, 3, 400)
ax[0].plot(grid, relu(grid)); ax[0].set_title("ReLU(z)"); ax[0].grid(alpha=0.3)
ax[1].plot(grid, relu_mask(grid)); ax[1].set_title("производная ReLU"); ax[1].grid(alpha=0.3)
for a in ax:
    a.axhline(0, c="k", lw=0.5); a.axvline(0, c="k", lw=0.5)
plt.tight_layout(); plt.show()

---
## 3. Инициализация и прямой проход

Каждый слой инициализируем своим масштабом: делим на $\sqrt{n_{in}}$, где
$n_{in}$ - число входов **этого** слоя. Для первого слоя это 784, для
второго 128. Логика та же, что в ноутбуке 02: не дать дисперсии
активаций вырасти при суммировании.

In [ ]:
n_hidden = 128

w1 = (torch.randn(d, n_hidden, generator=GEN) / math.sqrt(d)).requires_grad_()
b1 = torch.zeros(n_hidden, requires_grad=True)
w2 = (torch.randn(n_hidden, num_classes, generator=GEN) / math.sqrt(n_hidden)).requires_grad_()
b2 = torch.zeros(num_classes, requires_grad=True)

params = {"w1": w1, "b1": b1, "w2": w2, "b2": b2}
total_params = sum(p.numel() for p in params.values())

for name, p in params.items():
    print(f"{name}: {tuple(p.shape)}")
print(f"\nвсего обучаемых параметров: {total_params}")
print(f"у линейной модели было: {784 * 10 + 10}")


def forward(x):
    # x -> линейный -> ReLU -> линейный -> логиты
    z1 = x @ w1 + b1
    h1 = relu(z1)
    z2 = h1 @ w2 + b2
    return z2


bs = 64
xb, yb = x_train[:bs], y_train[:bs]

z2 = forward(xb)
loss = cross_entropy(z2, yb)

print(f"\nна необученной сети: ошибка {loss.item():.4f} "
      f"(ожидание log(10) = {math.log(10):.4f})")
print(f"точность на батче: {accuracy(z2, yb).item() * 100:.1f}%")

Проверим утверждение про эквивалентность линейных слоев численно: соберем
ту же сеть **без** ReLU и ту же сеть как один слой с матрицей $W_1 W_2$.
Выходы должны совпасть.

In [ ]:
with torch.no_grad():
    # два линейных слоя подряд, без нелинейности
    two_layers = (xb @ w1 + b1) @ w2 + b2
    # эквивалентный один слой
    one_layer = xb @ (w1 @ w2) + (b1 @ w2 + b2)

    print("max |два слоя без ReLU - один эквивалентный слой| =",
          (two_layers - one_layer).abs().max().item())
    print("то есть без ReLU глубина не дает ничего\n")

    with_relu = forward(xb)
    print("а с ReLU выходы уже другие: max |разность| =",
          (with_relu - two_layers).abs().max().item())

---
## 4. Обратный проход вручную: цепное правило по слоям

Прямой проход:

$$X \xrightarrow{\ W_1, b_1\ } Z_1 \xrightarrow{\ \mathrm{ReLU}\ } H_1 \xrightarrow{\ W_2, b_2\ } Z_2 \xrightarrow{\ \text{softmax+NLL}\ } L$$

Обратный идет строго в обратном порядке. Удобно думать о нем как о конвейере:
**каждый слой получает градиент по своему выходу и обязан выдать градиент
по своим параметрам и по своему входу**. Градиент по входу - это то, что
передается предыдущему слою.

Шаг за шагом:

| № | Формула | Форма | Откуда |
|---|---|---|---|
| 1 | $G_2 = \dfrac{\mathrm{softmax}(Z_2) - Y}{B}$ | $(B, 10)$ | главная формула из ноутбука 02 |
| 2 | $\dfrac{\partial L}{\partial W_2} = H_1^\top G_2$ | $(128, 10)$ | линейный слой, вход $H_1$ |
| 3 | $\dfrac{\partial L}{\partial b_2} = \mathrm{sum}(G_2, 0)$ | $(10,)$ | линейный слой |
| 4 | $\dfrac{\partial L}{\partial H_1} = G_2 W_2^\top$ | $(B, 128)$ | градиент по входу слоя 2 |
| 5 | $G_1 = \dfrac{\partial L}{\partial H_1} \odot [Z_1 > 0]$ | $(B, 128)$ | **через ReLU, поэлементно** |
| 6 | $\dfrac{\partial L}{\partial W_1} = X^\top G_1$ | $(784, 128)$ | линейный слой, вход $X$ |
| 7 | $\dfrac{\partial L}{\partial b_1} = \mathrm{sum}(G_1, 0)$ | $(128,)$ | линейный слой |

Новых идей тут ровно одна - пункт 5. Пункты 2-4 и 6-7 - та же функция
`grad_linear` из ноутбука 02, просто вызванная дважды с разными входами:
для второго слоя вход это $H_1$, для первого - $X$.

Вот почему библиотеки устроены именно так: раз каждый слой умеет принимать
градиент по выходу и отдавать градиент по входу, слои можно складывать
в любую цепочку, и обратный проход соберется сам. Это и делает autograd.

In [ ]:
def backward_mlp(x, y, w1, b1, w2, b2):
    # Все градиенты сети x -> линейный -> ReLU -> линейный -> softmax.
    # Возвращает словарь {"w1":..., "b1":..., "w2":..., "b2":...}.
    x = x.detach()

    # прямой проход: нужны промежуточные z1 и h1
    z1 = x @ w1.detach() + b1.detach()
    h1 = relu(z1)
    z2 = h1 @ w2.detach() + b2.detach()

    # 1. градиент по логитам
    g2 = grad_logits(z2, y)

    # 2-4. второй линейный слой; его вход - h1
    grad_w2, grad_b2, grad_h1 = grad_linear(h1, w2, g2)

    # 5. через ReLU: поэлементное умножение на маску
    g1 = grad_h1 * relu_mask(z1)

    # 6-7. первый линейный слой; его вход - x
    grad_w1, grad_b1, _ = grad_linear(x, w1, g1)

    return {"w1": grad_w1, "b1": grad_b1, "w2": grad_w2, "b2": grad_b2}


manual = backward_mlp(xb, yb, w1, b1, w2, b2)

print("формы ручных градиентов против форм параметров:")
for name in params:
    print(f"  {name}: градиент {tuple(manual[name].shape)}, "
          f"параметр {tuple(params[name].shape)}, "
          f"{'ok' if manual[name].shape == params[name].shape else 'НЕ СОВПАЛО'}")

### Сверка с autograd

In [ ]:
for p_ in params.values():
    p_.grad = None

cross_entropy(forward(xb), yb).backward()

print("параметр   max |ручное - autograd|   отн. масштаба   вердикт")
print("-" * 66)
all_ok = True
for name, p_ in params.items():
    diff = (manual[name] - p_.grad).abs().max()
    rel = (diff / p_.grad.abs().max()).item()
    ok = torch.allclose(manual[name], p_.grad, rtol=1e-4, atol=1e-7)
    all_ok &= ok
    print(f"{name:<10} {diff.item():>22.3e} {rel:>16.3e}   "
          f"{'совпало' if ok else 'РАСХОЖДЕНИЕ'}")

assert all_ok, "ручной backward разошелся с autograd"
print("\nцепное правило через ReLU выведено верно")

### Проверка пункта 5 отдельно: что делает маска ReLU

Самое интересное место - шаг через нелинейность. Посмотрим на него вблизи:
сколько нейронов скрытого слоя закрыто и какая доля градиента на них гибнет.

In [ ]:
with torch.no_grad():
    z1 = xb @ w1 + b1
    h1 = relu(z1)
    mask = relu_mask(z1)

    g2 = grad_logits(h1 @ w2 + b2, yb)
    _, _, grad_h1 = grad_linear(h1, w2, g2)
    g1 = grad_h1 * mask

print(f"нейронов скрытого слоя: {n_hidden}, примеров в батче: {bs}")
print(f"активаций всего: {mask.numel()}")
print(f"из них открыто (z1 > 0): {int(mask.sum())} "
      f"({mask.mean().item() * 100:.1f}%)")
print()
print(f"норма градиента ДО маски  (dL/dH1): {grad_h1.norm().item():.6f}")
print(f"норма градиента ПОСЛЕ маски (dL/dZ1): {g1.norm().item():.6f}")
print(f"\nградиент строго на закрытых позициях: {g1[mask == 0].abs().max().item()}")
print("то есть ровно ноль - через закрытый нейрон не проходит ничего")

# Сколько нейронов закрыты на ВСЕХ примерах батча - кандидаты в мертвые
never_open = int((mask.sum(dim=0) == 0).sum())
print(f"\nнейронов, закрытых на всех {bs} примерах батча: {never_open}")
print("их веса на этом шаге не изменятся вообще")

---
## 5. Обучение

Цикл тот же, что в ноутбуке 02, только параметров четыре вместо двух.
Сверку ручных градиентов с autograd оставляем включенной: она проверяет,
что формулы верны не только в начальной точке, но и на всей траектории
обучения, когда маска ReLU постоянно меняется.

In [ ]:
# Переинициализация, чтобы ячейку можно было запускать повторно
torch.manual_seed(0)
GEN = torch.Generator().manual_seed(0)

w1 = (torch.randn(d, n_hidden, generator=GEN) / math.sqrt(d)).requires_grad_()
b1 = torch.zeros(n_hidden, requires_grad=True)
w2 = (torch.randn(n_hidden, num_classes, generator=GEN) / math.sqrt(n_hidden)).requires_grad_()
b2 = torch.zeros(num_classes, requires_grad=True)
params = {"w1": w1, "b1": b1, "w2": w2, "b2": b2}


@torch.no_grad()
def evaluate(x, y, batch_size=1000):
    total_loss = total_correct = 0.0
    total = 0
    for xb_, yb_ in iterate_minibatches(x, y, batch_size, shuffle=False):
        z_ = forward(xb_)
        k = yb_.shape[0]
        total_loss += float(cross_entropy(z_, yb_)) * k
        total_correct += float(accuracy(z_, yb_)) * k
        total += k
    return total_loss / total, total_correct / total


lr = 0.1
epochs = 10
bs = 64
check_every = 200

hist_loss, hist_valid_acc, hist_valid_loss, hist_epoch = [], [], [], []
grad_checks, dead_fraction = [], []

base_loss, base_acc = evaluate(x_valid, y_valid)
print(f"до обучения: ошибка {base_loss:.4f}, точность {base_acc * 100:.2f}%\n")

step = 0
for epoch in range(1, epochs + 1):
    for xb, yb in iterate_minibatches(x_train, y_train, bs, shuffle=True, generator=GEN):
        loss = cross_entropy(forward(xb), yb)
        loss.backward()

        if step % check_every == 0:
            manual = backward_mlp(xb, yb, w1, b1, w2, b2)
            worst = 0.0
            for name, p_ in params.items():
                assert torch.allclose(manual[name], p_.grad, rtol=1e-3, atol=1e-7), \
                    f"ручной градиент {name} разошелся на шаге {step}"
                worst = max(worst, float((manual[name] - p_.grad).abs().max()
                                         / p_.grad.abs().max().clamp(min=1e-12)))
            grad_checks.append(worst)

            with torch.no_grad():
                m = relu_mask(xb @ w1 + b1)
                dead_fraction.append(1.0 - m.mean().item())

        with torch.no_grad():
            for p_ in params.values():
                p_ -= lr * p_.grad
                p_.grad.zero_()

        hist_loss.append(float(loss.detach()))
        step += 1

    v_loss, v_acc = evaluate(x_valid, y_valid)
    hist_epoch.append(epoch)
    hist_valid_loss.append(v_loss)
    hist_valid_acc.append(v_acc)
    print(f"эпоха {epoch:>2}/{epochs}: валидация ошибка {v_loss:.4f}, "
          f"точность {v_acc * 100:.2f}%")

print(f"\nсверок с autograd: {len(grad_checks)}, "
      f"худшее относительное расхождение {max(grad_checks):.2e}")

---
## 6. Результаты и сравнение с линейной моделью

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
window = 100
smooth = torch.tensor(hist_loss).unfold(0, window, 1).mean(dim=1)
ax.plot(hist_loss, alpha=0.2, lw=0.5, label="ошибка на батче")
ax.plot(range(window - 1, len(hist_loss)), smooth, color="crimson", lw=1.8,
        label=f"скользящее среднее ({window})")
ax.axhline(math.log(10), ls="--", c="gray", lw=1, label="log(10)")
ax.set_xlabel("шаг"); ax.set_ylabel("ошибка")
ax.set_title("Ошибка на обучающих батчах"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(hist_epoch, [a * 100 for a in hist_valid_acc], "o-", color="seagreen",
        label="MLP 784-128-10")
ax.axhline(92.43, ls="--", c="gray", label="линейная модель, 92.4%")
ax.set_xlabel("эпоха"); ax.set_ylabel("точность, %")
ax.set_title("Точность на валидации"); ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
ax.plot([i * check_every for i in range(len(dead_fraction))],
        [f * 100 for f in dead_fraction], color="darkorange")
ax.set_xlabel("шаг"); ax.set_ylabel("доля закрытых, %")
ax.set_title("Доля закрытых нейронов ReLU"); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

final_loss, final_acc = evaluate(x_valid, y_valid)
print(f"MLP:             ошибка {final_loss:.4f}, точность {final_acc * 100:.2f}%, "
      f"{total_params} параметров")
print(f"линейная модель: ошибка 0.2691, точность 92.43%, {784 * 10 + 10} параметров")
# Число ошибок берем прямым счетом, а не через округленную точность
with torch.no_grad():
    errors_mlp = int((torch.argmax(forward(x_valid), dim=1) != y_valid).sum())
print(f"\nошибок у линейной модели было 757, у MLP стало {errors_mlp} из 10000")

Правый график стоит прочитать отдельно: доля закрытых нейронов ReLU
не падает в нуль и не уходит в единицу, а устанавливается примерно на
половине. Это и есть рабочий режим - сеть использует ReLU как переключатель,
разные нейроны отвечают за разные группы примеров.

### Матрица ошибок: что исправилось

In [ ]:
with torch.no_grad():
    preds_valid = torch.argmax(forward(x_valid), dim=1)

cm = torch.zeros(10, 10, dtype=torch.int64)
for t, p_ in zip(y_valid.tolist(), preds_valid.tolist()):
    cm[t, p_] += 1

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("предсказание"); ax.set_ylabel("истина")
ax.set_title("Матрица ошибок MLP на валидации")
for i in range(10):
    for j in range(10):
        v = cm[i, j].item()
        if v:
            ax.text(j, i, v, ha="center", va="center", fontsize=7,
                    color="white" if v > cm.max() / 2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()

print("точность по классам (в скобках - линейная модель из ноутбука 02):")
linear_per_class = [96.9, 97.6, 90.3, 88.1, 94.6, 88.4, 95.7, 93.7, 88.4, 90.2]
for k in range(10):
    acc_k = cm[k, k].item() / cm[k].sum().item() * 100
    print(f"  {k}: {acc_k:5.1f}%   ({linear_per_class[k]:4.1f}%)   "
          f"{'+' if acc_k > linear_per_class[k] else '-'}"
          f"{abs(acc_k - linear_per_class[k]):.1f}")

### Чему научился скрытый слой

У линейной модели столбцы $W$ читались прямо как шаблоны цифр. Здесь первый
слой учит не цифры, а **промежуточные признаки**: штрихи, изгибы, замкнутые
области. Смотреть на них интереснее, но интерпретируются они хуже - это
нормально, именно поэтому глубокие сети называют менее прозрачными.

Каждая картинка ниже - столбец $W_1[:, j]$, то есть то, на что реагирует
один нейрон скрытого слоя.

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(16, 5))
w1_img = w1.detach()
limit = w1_img.abs().max()

# Берем нейроны с самой большой нормой веса - они "сильнее" участвуют в ответе
strength = w1_img.norm(dim=0)
top_neurons = torch.topk(strength, 30).indices

for ax, j in zip(axes.ravel(), top_neurons.tolist()):
    ax.imshow(w1_img[:, j].reshape(28, 28), cmap="bwr", vmin=-limit, vmax=limit)
    ax.set_title(f"нейрон {j}", fontsize=8)
    ax.axis("off")
fig.suptitle("Столбцы W1: признаки, которые выучил скрытый слой", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Примеры ошибок: на них видно, что осталось сложным
wrong = (preds_valid != y_valid).nonzero().flatten()
print(f"ошибок: {len(wrong)} из {len(y_valid)}")

fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for ax, idx in zip(axes.ravel(), wrong[:16].tolist()):
    with torch.no_grad():
        p_vec = softmax(forward(x_valid[idx:idx + 1]))[0]
    ax.imshow(x_valid[idx].reshape(28, 28), cmap="gray")
    ax.set_title(f"{y_valid[idx].item()} -> {preds_valid[idx].item()}\n"
                 f"p={p_vec[preds_valid[idx]]:.2f}", fontsize=9)
    ax.axis("off")
fig.suptitle("Ошибки MLP: истина -> предсказание (и уверенность модели)", y=1.10)
plt.tight_layout(); plt.show()

Большая часть оставшихся ошибок - это примеры, на которых и человек
засомневается: обрезанные, перекошенные или написанные нестандартно цифры.

---
## Итоги ноутбука 03

**Новая математика:**

1. Без нелинейности глубина не дает ничего: $(XW_1 + b_1)W_2 + b_2$
   сводится к одному слою. Проверено численно.
2. $\mathrm{ReLU}(z) = \max(0,z)$, производная - маска $[z > 0]$.
3. Цепное правило через поэлементную функцию - это поэлементное умножение
   на маску, а не матричное умножение:
   $\partial L/\partial Z_1 = (\partial L/\partial H_1) \odot [Z_1 > 0]$.
4. Полный backward: $G_2 \to (W_2, b_2, H_1) \to G_1 \to (W_1, b_1)$,
   то есть одна и та же `grad_linear`, вызванная дважды, плюс маска между ними.
5. Через закрытый нейрон градиент не проходит вообще - отсюда мертвые нейроны.

**Результат:** около 97.5% против 92.4% у линейной модели, при 101770
параметрах против 7850. Число ошибок на валидации упало примерно с 757 до 250.

**Дальше:** ноутбук `04_compare_with_nn.ipynb` - та же сеть через `nn.Module`
и `optim.SGD`, с построчным сопоставлением, что именно библиотека сделала
за нас.